# Phase 3 — Dataset Engineering

Creates `processed_v1` from the validated `cleaned_v1` dataset. It merges classes, produces statistics, class weights, Person analysis, configurable augmentation metadata, charts, and a final validation report. It does not modify `original/` or `cleaned_v1`, and it does not augment images or train a model.

In [1]:
from pathlib import Path
import json
import sys

ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(ROOT / 'src'))
from weapon_threat_detection.artifacts import configure_logger, utc_timestamp, write_json
from weapon_threat_detection.config import load_config
from weapon_threat_detection.engineering import (
    MERGED_CLASS_NAMES,
    analyze_person_class,
    build_augmentation_pipeline,
    build_quality_report,
    calculate_class_weights,
    collect_dataset_statistics,
    create_merged_dataset,
    save_publication_charts,
    write_yaml,
)
from weapon_threat_detection.validation import build_integrity_report, summarize_records, validate_dataset, write_validation_csv

config = load_config(ROOT / 'configs' / 'project.yaml')
source = ROOT / config['engineering']['source_dataset']
target = ROOT / config['engineering']['target_dataset']
reports_dir = ROOT / config['project']['reports_dir']
logger = configure_logger(ROOT / config['project']['logs_dir'], 'dataset_engineering')
if target.exists() and any(target.iterdir()):
    final_records = validate_dataset(target, config['dataset']['splits'], len(MERGED_CLASS_NAMES), config['dataset']['image_extensions'])
    existing_summary = summarize_records(final_records)
    if any(status not in {'valid', 'valid_background'} for status in existing_summary):
        raise RuntimeError(f'Existing processed_v1 is invalid and will not be overwritten: {existing_summary}')
    metadata = json.loads((target / 'metadata.json').read_text(encoding='utf-8'))
else:
    metadata = create_merged_dataset(source, target, config['dataset']['splits'])
statistics = collect_dataset_statistics(target, MERGED_CLASS_NAMES, config['dataset']['splits'])
weights = calculate_class_weights(statistics, MERGED_CLASS_NAMES)
person_analysis = analyze_person_class(target, person_id=4, splits=config['dataset']['splits'])
charts = save_publication_charts(statistics, person_analysis, reports_dir)
before_cleaning_path = max(reports_dir.glob('dataset_statistics_*.json'), key=lambda path: path.stat().st_mtime)
before_cleaning = json.loads(before_cleaning_path.read_text(encoding='utf-8'))
after_cleaning = collect_dataset_statistics(source, config['dataset']['class_names'], config['dataset']['splits'])
quality_report = build_quality_report(before_cleaning, after_cleaning, statistics, weights, person_analysis)
augmentation_path = ROOT / config['engineering']['augmentation_config']
pipeline = build_augmentation_pipeline(augmentation_path)
stamp = utc_timestamp()
statistics_path = write_json(reports_dir / f'processed_v1_statistics_{stamp}.json', statistics)
weights_json_path = write_json(reports_dir / 'class_weights.json', weights)
weights_yaml_path = write_yaml(reports_dir / 'class_weights.yaml', weights)
person_path = write_json(reports_dir / f'person_class_analysis_{stamp}.json', person_analysis)
quality_path = write_json(reports_dir / f'dataset_quality_report_{stamp}.json', quality_report)
augmentation_report = write_json(reports_dir / f'augmentation_pipeline_{stamp}.json', {'config': str(augmentation_path), 'enabled': False, 'apply_to_dataset': False, 'pipeline': repr(pipeline)})
final_records = validate_dataset(target, config['dataset']['splits'], len(MERGED_CLASS_NAMES), config['dataset']['image_extensions'])
final_summary = summarize_records(final_records)
if any(status not in {'valid', 'valid_background'} for status in final_summary):
    raise RuntimeError(f'processed_v1 validation failed: {final_summary}')
validation_path = write_validation_csv(final_records, reports_dir / f'processed_v1_validation_{stamp}.csv')
integrity_path = write_json(reports_dir / f'processed_v1_integrity_{stamp}.json', build_integrity_report(final_records, config['dataset']['splits']))
completion_path = write_json(reports_dir / f'phase_3_completion_{stamp}.json', {'metadata': metadata, 'statistics': str(statistics_path), 'class_weights_json': str(weights_json_path), 'class_weights_yaml': str(weights_yaml_path), 'person_analysis': str(person_path), 'quality_report': str(quality_path), 'charts': charts, 'augmentation_pipeline': str(augmentation_report), 'validation_records': str(validation_path), 'integrity_report': str(integrity_path), 'validation_summary': final_summary, 'source_modified': False, 'augmentation_applied': False, 'training_started': False})
logger.info('Created processed_v1 with merged classes: %s', final_summary)
logger.info('No augmentation or model training was performed.')
print({'processed_dataset': str(target), 'final_classes': MERGED_CLASS_NAMES, 'statistics': statistics, 'validation': final_summary, 'completion_report': str(completion_path)})

/Users/mohamedtarek/weapon130/.venv/lib/python3.10/site-packages/albumentations/__init__.py:24: UserWarning: A new version of Albumentations is available: 2.0.8 (you have 1.4.24). Upgrade using: pip install -U albumentations. To disable automatic update checks, set the environment variable NO_ALBUMENTATIONS_UPDATE to 1.
  check_for_updates()


{'processed_dataset': '/Users/mohamedtarek/weapon130/processed/processed_v1', 'final_classes': ['Handheld_Weapon', 'Explosive', 'Fire_Smoke', 'Firearm', 'Person'], 'statistics': {'total_images': 130245, 'total_annotations': 293130, 'background_images': 12342, 'annotations_per_class': {'Handheld_Weapon': 61168, 'Explosive': 25288, 'Fire_Smoke': 77860, 'Firearm': 117069, 'Person': 11745}, 'percentage_per_class': {'Handheld_Weapon': 20.86719203083956, 'Explosive': 8.626889093576228, 'Fire_Smoke': 26.561593832088153, 'Firearm': 39.93757036127315, 'Person': 4.006754682222905}, 'splits': {'train': {'images': 104253, 'annotations': 244686, 'background_images': 9032}, 'valid': {'images': 13158, 'annotations': 21641, 'background_images': 1780}, 'test': {'images': 12834, 'annotations': 26803, 'background_images': 1530}}}, 'validation': {'valid': 117903, 'valid_background': 12342}, 'completion_report': '/Users/mohamedtarek/weapon130/reports/phase_3_completion_20260723T092600Z.json'}
